<img src="https://www.epfl.ch/about/overview/wp-content/uploads/2020/07/logo-epfl-1024x576.png" width="140px" alt="EPFL_logo">

## Image Processing Laboratory Notebooks
---

This Jupyter Notebook is part of a series of computer laboratories that are designed
to teach image-processing programming; they are running on the EPFL's Noto server. They are the practical complement of the theoretical lectures of the EPFL's Master course 
[**MICRO-512 Image Processing II**](https://moodle.epfl.ch/course/view.php?id=463) taught by Prof. M. Unser and Prof. D. Van de Ville.

The project is funded by the Center for Digital Education and the School of Engineering. It is owned by the [Biomedical Imaging Group](http://bigwww.epfl.ch/). 
The distribution or reproduction of the notebook is strictly prohibited without the written consent of the authors.  &copy; EPFL 2026.

**Authors**: 
    [Pol del Aguila Pla](mailto:pol.delaguilapla@epfl.ch), 
    [Kay Lächler](mailto:kay.lachler@epfl.ch),
    [Alejandro Noguerón Arámburu](mailto:alejandro.nogueronaramburu@epfl.ch), and
    [Daniel Sage](mailto:daniel.sage@epfl.ch).
    
To ensure your work is graded correctly by our automated system, **do not create new cells or delete/rearrange/copy existing ones** when you submit. The current cells contain hidden metadata required for the auto-grader to identify your solutions. If you create temporary cells for testing during your work, remember to clean them up before submission.

# Lab 4.1: Orientation Warmup
**Released**: Thursday, February 19, 2026

**Submission deadline**: Monday, March 2, 2026, before 23:59 on [Moodle](https://moodle.epfl.ch/course/view.php?id=463)

**Grade weight**: Lab 4 (18 points), 7.5 % of the overall grade

**Help Session**: Thursday, February 26, 2026

**Related lectures**: Chapter 6

### Student Name: 
### SCIPER: 

Double-click on this cell and fill your name and SCIPER number. Then, run the cell below to verify your identity in Noto and set the seed for random results.

In [ ]:
import getpass
# This line recovers your camipro number to mark the images with your ID
uid = int(getpass.getuser().split('-')[2]) if len(getpass.getuser().split('-')) > 2 else ord(getpass.getuser()[0])
print(f'SCIPER: {uid}')

## Imports
In the next cell we import Python libraries we will use throughout the lab, as well as the `IPLabViewer` class, created specifically for this course, which provides interactive image visualization based on the `ipywidgets` library:
* [`matplotlib.pyplot`](https://matplotlib.org/3.2.2/api/_as_gen/matplotlib.pyplot.html), to display images,
* [`ipywidgets`](https://ipywidgets.readthedocs.io/en/latest/), to make the image display interactive,
* [`numpy`](https://numpy.org/doc/stable/reference/index.html), for mathematical operations on arrays,
* [`cv2`](https://docs.opencv.org/2.4/index.html), for image processing tasks.

We will then load the `ImageViewer` class (see the documentation [here](https://github.com/Biomedical-Imaging-Group/interactive-kit/wiki/Image-Viewer) or run the Python command `help(viewer)` after loading the class).

Finally, we load the images you will use in the exercise to test your functions. 

In [ ]:
# Configure plotting as dynamic
%matplotlib widget

# Import standard required packages for this exercise
import math
import sys
import time
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
import cv2 as cv
import scipy.signal
from interactive_kit import imviewer as viewer 

# Load images to be used in this exercise 
dendrochronology = cv.imread('images/dendrochronolgy.tif', cv.IMREAD_UNCHANGED).astype('float64')

# Orientation Warmup Laboratory (6 points)

Lab 4 will introduce you to the implementation of image processing algorithms and systems that rely on directional analysis, i.e., on the orientation features of an image.
Note that to obtain orientation features, one mainly uses linear filtering, which was covered in [Lab 2: Filtering](../2_filtering_lab/1_filtering.ipynb) of [Image Processing I](https://moodle.epfl.ch/enrol/index.php?id=522).

In this warm-up notebook, we take the opportunity for you to (re-)familiarize yourself with the tools we will use in the upcoming labs, like Python, the [`ImageViewer`](https://github.com/Biomedical-Imaging-Group/interactive-kit/wiki/Image-Viewer) class, and [`openCV` (cv2)](https://docs.opencv.org/2.4/index.html). Moreover, you will learn some *advanced* or *efficient* filtering techniques. 

In short, you will implement an efficient approximation of the Gaussian smoothing filter, which you have seen in [Image Processing I](https://moodle.epfl.ch/course/view.php?id=522) and most likely in other subjects. If you didn't take Image Processing I and/or are unsure about some functionalities, be sure to check out [Lab 0: Introductory](../0_introductory_lab/introductory.ipynb), where all the basic tools are thoroughly introduced. If you still have questions concerning either programming or the lab in general, don't hesitate to contact one of the TAs listed on [Moodle](https://moodle.epfl.ch/course/view.php?id=463).

## Efficient Gaussian smoothing

Gaussian smoothing is a fundamental part of many image processing algorithms.
It is most often used to suppress noise, which improves the reliability of any further processing.
If you took Image Processing I, you already implemented a separable Gaussian smoothing algorithm in [Lab 2: Filtering](../2_filtering_lab/2_filtering_applications.ipynb), which is provided in the next cell.
This function is significantly faster to run than its non-separable version, but it can still take a pretty long time to complete when using large values of $\sigma$.
Therefore, we will now implement an approximation of the Gaussian smoothing function that has a runtime which is independent of $\sigma$.

First, run the cell below to define the separable Gaussian smoothing function. We'll need it to compare the execution time.

**Note:** If you don't remember how the Gaussian filter works, look it up [here](https://en.wikipedia.org/wiki/Gaussian_blur). In any case, read through the code provided below and make sure you understand what each line is doing.

In [ ]:
def gaussian(image, sigma):
    # generate mask
    # size (truncate with 3 sigma)
    N = 2 * np.ceil(3 * sigma) + 1
    # define mask and fill it's values
    mask = np.zeros(N)
    # precompute values 
    C = 1 / (np.sqrt(2 * np.pi) * sigma)
    center = math.floor(N / 2)
    two_variance = 2 * sigma ** 2
    # fill the mask
    for x in range(N):
        mask[x] = C * np.exp(-(x - center)**2) / two_variance
    # normalize the mask values to sum 1
    mask /= mask.sum()
    
    convolved = np.zeros_like(image)
    for i in range(image.shape[0]):
        convolved[i] = filter1d(image[i], mask)

    for j in range(image.shape[1]):
        convolved[:, j] = filter1d(convolved[:, j], mask)

    return convolved
    

def filter1d(image, kernel):
    padding = kernel.shape[0] // 2
    padded = np.pad(image, padding, mode='reflect')
    convolved = np.zeros_like(image)
    for i in range(convolved.shape[0]):
        for j in range(kernel.shape[0]):
            convolved[i] += padded[i + j] * kernel[-j - 1]
    return convolved

There are several possibilities to approximate a Gaussian filter, one of which is a cascade of $N$ symmetric exponential filters. As you might remember from Image Processing I, a symmetric exponential filter is defined by its transfer function:
$$\operatorname{H}_a(z) = \frac{C_a}{(1-az^{-1})(1-az)}$$
and is determined by its pole $a$.

The equivalent standard deviation $\sigma_{eq}$ of cascading $N$ exponential filters is related to the pole value $a$ by:
$$\sigma_{eq}^2 = \frac{2Na}{(1-a)^2}$$

### Calculate the poles (1 point)

Your first task in this exercise, **for 1 point**, is to implement the function `get_pole(sigma, N)`, which calculates the pole value $a$ from `sigma` ($\sigma_{eq}$) and `N`.

**Note:** Check *chapter 3.4 of IP1* for more details on the symmetric exponential filter and other methods for efficient Gaussian filtering.

**Hint:** You might want to use the Numpy square root function `np.sqrt()`.

In [ ]:
def get_pole(sigma, N):
    # initialize the pole value
    a = 0
    
    # YOUR CODE HERE

    return a

Run the next cell for a simple sanity check. Note that passing a sanity check does not necessarily mean that your function is correct!

In [ ]:
err = False;
if get_pole(10, 5).round(4) != 0.7298:
    print('WARNING!\nThe get_pole function is not correct for sigma=10, N=5. Your value: ' + get_pole(10, 5).round(4) + ', expected value: ' + 0.7298)
    err = true
if get_pole(3, 2).round(4) != 0.5195:
    print('WARNING!\nThe get_pole function is not correct for sigma=3, N=2. Your value: ' + get_pole(3, 2).round(4) + ', expected value: ' + 0.5195)
    err = true
if err is False:
    print('Well done, your get_pole function passed the sanity check.')

### Implement the symmetric exponential filter (2 points)

<table><tr>
<td> 
  <p align="center" style="padding: 10px">
    <img alt="Separated exponential filter" src="images/exponential_filter_separation.png" width="500">
    <br>
    <em style="color: grey">Separable implementation of the symmetric exponential filter.</em>
  </p> 
</td>
</tr></table>

Now that we have the correct pole value corresponding to the $\sigma$ that we want for the blurring, we can implement the symmetric exponential filter using its difference equation. To simplify this task, we can first separate the transfer function into its causal and anti-causal components (see the image above). Using this separation, it is easy to devise a recursive-filtering algorithm that consists of 3 steps:

1. Normalization: $\;y_1[k] = C_a \cdot x[k]$
2. Causal filtering: $\;y_2[k] = y_1[k] + a\,y_2[k-1], \; \mbox{for} \, (k=1,\ldots,N-1)$
3. Anti-Causal filtering: $\;y[k] = a\,(\,y[k+1] - y_2[k]\,), \; \mbox{for} \, (k=N-2,\ldots,0)$

with $N$ the length of the signal and $C_a$ given by:
$$C_a = -\frac{(1-a)^2}{a}$$

In the cell below, **for 2 points**, complete the function `sym_exp_filter(signal, a , N)` and implement the recursive-filtering algorithm proposed above to perform $N$ cascaded symmetric exponential filters on the input `signal`.

As you may have noticed, it is not trivial to figure out what the initial values $y_2[0]$ and $y[N-1]$ should be in each of the filtering steps. Since this topic is too advanced for this course, we will provide you with the two functions `get_initial_causal_coefficient(signal, a)` and `get_initial_anticausal_coefficient(signal, a)`, which calculate the appropriate initial conditions for you (this step is already implemented).

**Notes:**
- If you check *page 3-48* of the IP1 course notes, you will see that the two proposed separations are not the same. We use this specific separation because it simplifies the calculation of the initial condition, which is essential for the correct behavior of the filter.
- The input parameter `signal` is an `Image` object of size $(nx \times 1)$, so it only contains one single row. You can access the elements of this row with <code>signal.getPixel(x, 0)</code>.
- The most efficient way of implementing the normalization part of the filter would be to normalize the image once with an adjusted normalization value $C_a^\prime = C_a^N$ instead of normalizing it $N$ times with $C_a$, which works because the normalization is a multiplication by a constant. You can, of course, do this if you want, but both methods will be equally accepted.

In [ ]:
# function that applies N successive symmetric exponential filters with pole value a to the signal
def sym_exp_filter(signal, a, N):
    # initialize the output signal as a copy of the input signal
    output = signal.copy()
    
    # perform N successive exponential filtering steps
    for n in range(N):
        # Normalization
        # YOUR CODE HERE
        
        # Set the causal initial condition
        output[0] = get_initial_causal_coefficient(output, a)
        
        # Perform causal filtering
        # YOUR CODE HERE
        
        # Set the anti-causal initial condition
        output[-1] = get_initial_anticausal_coefficient(output, a)
        
        # Perform anti-causal filtering
        # YOUR CODE HERE
    return output

# --- DO NOT CHANGE THESE FUNCTIONS! ---
# Calculates the initial condition of the causal filter
def get_initial_causal_coefficient(signal, a):
    n = signal.shape[0]
    z1 = a
    zn = a ** (n - 1)
    summ = signal[0] + zn * signal[-1]
    horizon = 2 + int(np.log(1e-6) / np.log(np.abs(a)))
    horizon = min(horizon, n)
    #print(n, z1, zn, summ, horizon)
    zn = zn * zn
    for l in range(1, horizon-1):
        zn = zn / a
        summ = summ + (z1 + zn) * signal[l]
        z1 = z1 * a
    return summ / (1.0 - (a ** (2 * n - 2)))

# Calculates the initial condition of the anti-causal filter
def get_initial_anticausal_coefficient(signal, a):
    return (a * signal[-2] + signal[-1]) * a / (a * a - 1)

Now run the cell below to perform a quick sanity check again.

In [ ]:
# create test signal
test_signal = np.zeros(11)
test_signal[5] = 1
# apply the function to the test image with a pre-calculated pole value
test_signal_filtered = sym_exp_filter(test_signal, a=0.4514162296451364, N=3)
# test symmetry of the filtered signal
err = False
for k in range(int(test_signal_filtered.shape[0]/2)):
    if test_signal_filtered[k].round(4) != test_signal_filtered[-1-k].round(4):
        print(test_signal_filtered)
        print('WARNING!\nThe output of your sym_exp_filter is not symmetric!')
        err = True
        break
# test against precomputed values
expected_output = np.array([0.0543, 0.0608, 0.0798, 0.1095, 0.1423, 0.1610, 0.1423, 0.1095, 0.0798, 0.0608, 0.0543])
if not np.allclose(test_signal_filtered.round(4), expected_output, rtol=1e-4):
    print('WARNING!\nThe sym_exp_filter is not yet correct for sigma=3 and N=3!.\nInput signal:\n', test_signal, 
                'Your output:\n', test_signal_filtered, 'Expected output:\n', expected_output)
    err = True
if err == False:
    print('Well done, your sym_exp_filter passed the sanity check.')

Before we let you implement the complete smoothing pipeline, let's first do a comparison between the effect of the symmetric exponential filter and the Gaussian smoothing. In the next cell, we will apply your `sym_exp_filter` function to an impulse signal using different values for `sigma` and `N`. In the cell below, we will then apply the Gaussian filter to the same impulse signal and create an interactive plot that shows the effect of both filters, as well as the error between the two. Run the next two cells and experiment with the interactive slider to explore the differences.

In [ ]:
sigmas = [1, 2, 3, 4, 5]
Ns = [1, 2, 3, 4, 5]

test_signals = np.zeros((len(sigmas) * len(Ns), 31))
test_signals[:, int(test_signals.shape[1]/2)] = 1
test_signals_smoothed = np.zeros_like(test_signals)

for i, sigma in enumerate(sigmas):
    for j, N in enumerate(Ns):
        index = i * len(Ns) + j
        pole = get_pole(sigma, N)
        test_signals_smoothed[index] = sym_exp_filter(test_signals[index], pole, N)

In [ ]:
# Declare slider for sigma
sigma_slider = widgets.IntSlider(value=1, min=1, max=5, description='\u03C3') 

# Initialize figure
plt.close('all')
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

# Function that calculates the MSE (in dB) between the ideal Gaussian impulse response and the impulse response of the box-filter approximation
def calc_err(c, g):
    # Calculate MSE (in dB) and return result
    return 10*np.log10(np.mean((c-g)**2))

# Function that generates an ideal Gaussian impulse response with sigma sig
def gen_ideal_gaussian(sig):
    # Calculate size of Gaussian (see Lab 2 or IP1 course notes)
    M = 2 * np.ceil(3*sig) + 1
    # Get signal
    g = scipy.signal.windows.gaussian(31, sig)
    # Define array with the indeces
    x_g = np.linspace(-g.shape[0]//2+1, g.shape[0]//2, g.shape[0])
    return x_g, g / np.sum(g)

# Plotting function - Callback for slider
def plot_approximation(change):
    # Get value of sigma, initialize variables of interest and clear axes
    sigma = change.new
    errors = []
    legends = []
    axs[0].clear()
    axs[1].clear()
    # Generate ideal Gaussian impulse response with the desired sigma and plot
    x_g, gauss_ideal = gen_ideal_gaussian(sigma)
    axs[0].plot(x_g, gauss_ideal,'k')
    # Iterate through box filters
    for i in range(5):
        # Calculate error and append to corresponding variable
        errors.append(calc_err(test_signals_smoothed[(sigma-1)*5 + i], gauss_ideal))
        # Plot box approximation and add legend
        legends.append(f'Exp filter (N={i + 1})')
        axs[0].plot(x_g, test_signals_smoothed[(sigma-1)*5 + i], alpha=0.5)
    # Format plot
    axs[0].legend(['Ideal Gaussian'] + legends); axs[0].set_xlabel('x'); axs[0].set_ylabel('h[n]')
    axs[0].set_title(rf'Gaussian $\sigma$={sigma} vs Exponential filter approximations'); axs[0].grid()
    axs[0].set_xlim([-(5+sigma*2), 5+sigma*2])
    axs[1].plot(list(range(1, 6)), errors)
    axs[1].set_xticks(list(range(1, 6)))
    axs[1].set_xlabel('N'); axs[1].set_ylabel('MSE [dB]')
    axs[1].set_title(rf'MSE, Gaussian $\sigma$={sigma} vs Exponential filter approximations'); axs[1].grid()
    fig.tight_layout()

# Didplay widget and link to callback
display(sigma_slider)
sigma_slider.observe(plot_approximation, 'value')
sigma_slider.value = 3

### Multiple choice question (1 point)
For **0.5 points each**, once you have investigated the differences between the effect of the symmetric exponential filter and the Gaussian smoothing, specify if the following statements are true or false:

1. As the number $N$ of the exponential filters increases, the approximation error to the Gaussian decreases across all values of $\sigma$.
2. As the value $\sigma$ increases, the approximation error of the exponential filters to the Gaussian increases across all values of $N$.

Modify the variable `statement1` and `statement2` to `True` or `False` in the next cell to reflect your answer.

In [ ]:
# Assign your answer to this variable
statement1 = None
statement2 = None
# YOUR CODE HERE

In [ ]:
# Check that the answer is in the valid range
assert statement1 in [True, False], 'Possible answers are True, False.'


In [ ]:
# Check that the answer is in the valid range
assert statement2 in [True, False], 'Possible answers are True, False.'


### Implement the Gaussian filter approximation (2 points)

Finally, we can use both previously defined functions to implement the complete Gaussian filter approximation. **For 2 points**, implement the function `smoothing(img, sigma, N)`, which approximates a Gaussian filter with standard deviation `sigma` using a cascade of `N` symmetric exponential filters, using the functions `get_pole` and `sym_exp_filter`.

**Note:** The filtering should be implemented in a separable manner, meaning you should first filter only the columns and then only the rows (or vice versa) instead of filtering the entire image at once.

In [ ]:
# function that approximates a Gaussian smoothing of std=sigma using a cascade of N exponential filters
def smoothing(img, sigma, N):
    out = np.zeros_like(img)
    
    # Get pole value
    # YOUR CODE HERE
    
    # Filter columns
    # YOUR CODE HERE
    
    # Filter rows
    # YOUR CODE HERE
    
    return out

As usual, run the next cell for a simple check on your function.

In [ ]:
# create test image
test_img = np.zeros((5, 5))
test_img[2, 2] = 1
# define the smoothing variables
sigma = 1.5
N = 3
# apply the smoothing to test_img
smoothed_test_img = smoothing(test_img, sigma, N)
# check that the blurring is isotropic
if np.allclose(smoothed_test_img, test_img, rtol=1e-4):
    print('WARNING!\nThe input image was not modified.')
elif not np.allclose(smoothed_test_img[2,:], smoothed_test_img[:,2], rtol=1e-4):
    print('WARNING!\nThe blurring is not isotropic.')
else:
    print('Good, the blurring is isotropic.')

In [ ]:
# precomputed correct output
correct_output = np.array([[0.0298, 0.0404, 0.0621, 0.0404, 0.0298], [0.0404, 0.0548, 0.0841, 0.0548, 0.0404],
                                [0.0621, 0.0841, 0.1291, 0.0841, 0.0621], [0.0404, 0.0548, 0.0841, 0.0548, 0.0404],
                                [0.0298, 0.0404, 0.0621, 0.0404, 0.0298]])
# check that the blurring is correct
if not np.allclose(smoothed_test_img.round(4), correct_output):
    print('WARNING!\nThe output on the test image is not correct.\nInput image:\n', test_img, 
            'Your output:\n', smoothed_test_img, 'Expected output:\n', correct_output)
else:
    print('Well done, your smoothing function passed the sanity checks!')


### Comparison

Now we have everything we need to compare the exponential filter approximation method to the usual separable Gaussian filter. First, we will compare the runtime of both methods as a function of the standard deviation $\sigma$ of the Gaussian.

The cell below applies both smoothing methods to multiple $\sigma$ values on the image `dendrochronology`, which is a $512 \times 512$ pixel image. Run the cell and wait for it to finish. Then, run the cell below to visualize the runtime comparison results.

Change the `sigma` values (or `N`) if you want, but beware that large `sigma`'s (and large `N`'s) can take a lot of time to run! **So remember to revert them to a small value before you submit the lab!**

**Remember:** A cell is still running if you see `In [*]` in the top-left corner. Once a number replaces the asterisk (*), the cell has finished running.

In [ ]:
# define sigma values 
sigmas = [1, 2, 4, 8, 16]
# define N
N = 3
dend_img = dendrochronology
errors = []
time_fast = []
time_norm = []

# run the Gaussian smoothing for all sigma values and measure the runtime of both methods
message = 'Running for σ = '
for sigma in sigmas:
    message = message + str(sigma) + ", "
    sys.stdout.write(message + "...\r")
    sys.stdout.flush()
    
    start_time = time.time()
    dend_smooth_img = smoothing(dend_img, sigma, N)
    end_time = time.time()
    time_fast.append(end_time - start_time)
    
    start_time = time.time()
    dend_smooth_norm_img = gaussian(dend_img, sigma)
    end_time = time.time()
    time_norm.append(end_time - start_time)

message = message[:-2] + ".     "
print(message)
print('Finished running for all the σ values.')

In [ ]:
# Plotting the runtime comparison results
plt.close('all') 
plt.figure(figsize=(12, 4))
# Plot runtime
plt.subplot(121)
plt.plot(sigmas, time_norm, marker='o')
plt.plot(sigmas, time_fast, marker='s')
plt.legend(['Separable Gaussian', 'Exponential filter cascade'])
plt.xlabel(r'$\sigma$'); plt.ylabel('Runtime [s]')
plt.grid(); plt.title('Runtime comparison')
# Plot improvement factor
plt.subplot(122)
plt.plot(sigmas, np.array(time_norm)/np.array(time_fast),marker='^')
plt.xlabel(r'$\sigma$');
plt.grid(); plt.title('Speed improvement factor')
plt.show()

If you implemented the functions `sym_exp_filter` and `smoothing` correctly, you should see in the plot above that the runtime of the exponential filter stays more or less constant, no matter the value of $\sigma$. On the other hand, the runtime of the separable Gaussian increases linearly with the value of $\sigma$. If we had compared the exponential filter to a non-separable version of the Gaussian filter, then you would probably still be waiting for the results, because the runtime of the non-separable Gaussian increases proportionally to $\sigma^2$.

Finally, we should compare the accuracy of our method to one of the Gaussian blurs provided by the Python library. Run the next two cells to apply both your function `smoothing` (with `N=3`) as well as `cv.GaussianBlur` on the `dendrochronology` image with a large range of $\sigma$ values and plot the average pixel error.

In [ ]:
sigmas = [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 22, 26, 30];

outs = []
dend_img = dendrochronology
message = 'Running for σ = '
for sigma in sigmas:
    message = message + str(sigma) + ", "
    sys.stdout.write(message + "...\r")
    sys.stdout.flush()
    
    dend_smooth_img = smoothing(dend_img, sigma, 3)
    outs.append(dend_smooth_img)

message = message[:-2] + ".     "
print(message)
print('Finished running for all the σ values.')

In [ ]:
# Visualize the average pixel error
errs = []
outs_cv =  []
for i, sigma in enumerate(sigmas):
    # Apply OpenCV Gaussian blur
    outs_cv.append(cv.GaussianBlur(dendrochronology, ksize=(0,0), sigmaX=sigma, borderType=cv.BORDER_REFLECT))
    # Calculate average pixel error
    errs.append(np.mean(np.abs(outs_cv[i][outs_cv[i] > 0] - np.array(outs[i])[outs_cv[i] > 0])/outs_cv[i][outs_cv[i] > 0]) * 100)
# Display result
plt.close('all')
plt.figure('Average pixel error')
plt.plot(sigmas, errs); plt.xlabel(r'$\sigma$'); plt.ylabel('Avg pixel error (%)')
plt.title('Average error per pixel'); plt.grid()
plt.show()

As you can see from the plot above, the exponential filter cascade is not perfect in approximating the Gaussian filter. But for large $\sigma$ values, the benefit of having a constant run-time usually outweighs this imperfection especially when the error is less than $5\%$, which is not a problem for many applications.

Finally run the cell below to visually compare the smoothed images.

In [ ]:
# Change the sigma index to see the result for different sigmas
sigma_idx = 1
# Give information to user
print(f'Average pixel error for σ = {sigmas[sigma_idx]}: {errs[sigma_idx]:.4}%')
plt.close('all')
img_list = [dendrochronology, outs_cv[sigma_idx], np.array(outs)[sigma_idx]]
title_list = ['Original Image', rf'Gaussian smoothed $\sigma$={sigmas[sigma_idx]}', rf'Exponential filter smoothed $\sigma$={sigmas[sigma_idx]}']
view = viewer(img_list, title=title_list, subplots=(1,3))

As a final note, as you may have observed during the comparison between your efficient Gaussian smoothing and OpenCV's Gaussian blur, OpenCV's version is still faster than your implementation. To completely understand how OpenCV manages to speed up the Gaussian filter goes beyond the scope of this course. Still, it has to do with the use of low-level programming languages that can utilize performance primitives offered by the CPU. Performance primitives are specialized CPU-specific computational instructions that leverage the CPU's architecture to perform calculations significantly faster.

For simplicity and time reasons, we will use `cv.GaussianBlur` as our Gaussian filter for the upcoming labs. Still, if you ever need to implement a fast Gaussian smoothing and don't have access to a library like OpenCV, you now know how to start.

🎉 Congratulations on finishing the first part of the orientation lab! Your next step is to complete the main lab, [Lab 4.2: Orientation](./2_orientation.ipynb), which focuses on the actual applications of the algorithms you will implement.

Make sure to save your notebook (you might want to keep a copy on your personal computer) and upload it to Moodle, **in a zip file with the other notebook of this lab.**

* Keep the name of the notebook as: *1_orientation_warmup.ipynb*,
* Name the `zip` file: *orientation_lab.zip*.